In [ ]:
from enum import Enum
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import csv

In [ ]:
class OptionBasicType(Enum):
    CALL = "CALL"
    PUT = "PUT"


class OptionExerciseType(Enum):
    EUROPEAN = "EUROPEAN"
    AMERICAN = "AMERICAN"


class EFD:

    def __init__(
        self,
        sigma_min: float,
        sigma_max: float,

        option_basic_type: OptionBasicType,
        option_exercise_type: OptionExerciseType,

        expiry_date: datetime,
        T0: datetime,

        strike_price: float,

        barrier_price: float = None,

        r: float = 0.05,

        delta_t: float = 0.001,
        delta_s: float = 1.0,

        dividend_amount: float = 0.0,
        dividend_date: datetime = None,

        verbose: bool = True,
    ) -> None:

        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

        self.option_basic_type = option_basic_type
        self.option_exercise_type = option_exercise_type

        self.expiry_date = expiry_date
        self.T0 = T0

        self.strike_price = strike_price
        self.barrier_price = barrier_price

        self.r = r

        self.delta_t = delta_t
        self.delta_s = delta_s

        self.dividend_amount = dividend_amount
        self.dividend_date = dividend_date

        self.verbose = verbose

        self.initialize_grid()


    def payoff(self, S):

        if self.option_basic_type == OptionBasicType.CALL:
            return np.maximum(S - self.strike_price, 0.0)

        elif self.option_basic_type == OptionBasicType.PUT:
            return np.maximum(self.strike_price - S, 0.0)
        else:
            raise ValueError("Invalid option type")


    def initialize_grid(self):

        T = (self.expiry_date - self.T0).days / 365
        self.M = int(T / self.delta_t)
        if self.M < 1:
            self.M = 1

        if self.barrier_price is None:
            S_max = 3.0 * self.strike_price
            self.N = int(S_max / self.delta_s)
            self.S_values = np.arange(self.N + 1) * self.delta_s

        else:
            if self.option_basic_type == OptionBasicType.CALL:
                self.N = int(self.barrier_price / self.delta_s)
                self.S_values = (
                    np.arange(self.N + 1) * self.delta_s
                )

            else:
                S_max = max(
                    3.0 * self.strike_price,
                    2.0 * self.barrier_price
                )

                self.N = int(
                    (S_max - self.barrier_price) / self.delta_s
                )

                self.S_values = (
                    self.barrier_price
                    + np.arange(self.N + 1) * self.delta_s
                )


        if self.dividend_date is not None:
            div_time = (
                self.dividend_date - self.T0
            ).days / 365

            self.dividend_step = int(div_time / self.delta_t)
        else:
            self.dividend_step = None

        self.grid = np.zeros((self.N + 1, self.M + 1))

        if self.option_exercise_type == OptionExerciseType.AMERICAN:
            self.exercise_grid = np.zeros((self.N + 1, self.M + 1))

        if self.verbose:
            print(f"Grid shape: {self.grid.shape}")

        self.grid[:, 0] = self.payoff(self.S_values)

        if self.barrier_price is None:
            if self.option_basic_type == OptionBasicType.CALL:
                self.grid[0, :] = 0.0
                self.grid[self.N, :] = (
                    self.S_values[-1]
                    - self.strike_price
                    * np.exp(-self.r * self.delta_t * np.arange(self.M + 1))
                )

            else:
                self.grid[self.N, :] = 0.0
                self.grid[0, :] = (
                    self.strike_price
                    * np.exp(-self.r * self.delta_t * np.arange(self.M + 1))
                )

        else:
            if self.option_basic_type == OptionBasicType.CALL:
                self.grid[0, :] = 0.0
                self.grid[self.N, :] = 0.0
            else:
                self.grid[0, :] = 0.0
                self.grid[self.N, :] = 0.0

    def apply_discrete_dividend(
        self,
        column,
        dividend_amount
    ):
        shifted_values = np.interp(
            self.S_values - dividend_amount,
            self.S_values,
            column,
            left=column[0],
            right=column[-1]
        )

        return shifted_values

    def calculate_grid(self):

        S_arr = self.S_values[1:-1]
        i_eff = S_arr / self.delta_s

        for j in range(1, self.M + 1):

            prev_col = self.grid[:, j - 1]
            gamma = (
                prev_col[2:]
                - 2.0 * prev_col[1:-1]
                + prev_col[:-2]
            ) / (self.delta_s ** 2)
            
            sigma = np.where(
                gamma > 0,
                self.sigma_max,
                self.sigma_min
            )

            p_u = 0.5 * self.delta_t * (
                sigma**2 * i_eff**2
                + self.r * i_eff
            )
            p_m = 1.0 - self.delta_t * (
                sigma**2 * i_eff**2
                + self.r
            )
            p_d = 0.5 * self.delta_t * (
                sigma**2 * i_eff**2
                - self.r * i_eff
            )

            self.grid[1:-1, j] = (
                p_u * prev_col[2:]
                + p_m * prev_col[1:-1]
                + p_d * prev_col[:-2]
            )

            if (
                self.dividend_step is not None
                and j == self.dividend_step
            ):
                self.grid[:, j] = self.apply_discrete_dividend(
                    self.grid[:, j],
                    self.dividend_amount
                )
                if self.barrier_price is not None:
                    if self.option_basic_type == OptionBasicType.CALL:
                        knocked = (
                            self.S_values >= self.barrier_price
                        )

                        self.grid[knocked, j] = 0.0
                    else:

                        knocked = (
                            self.S_values <= self.barrier_price
                        )

                        self.grid[knocked, j] = 0.0

            if self.option_exercise_type == OptionExerciseType.AMERICAN:
                intrinsic = self.payoff(S_arr)
                self.exercise_grid[1:-1, j] = np.where(
                    self.grid[1:-1, j] < intrinsic,
                    1,
                    0
                )

                self.grid[1:-1, j] = np.maximum(
                    self.grid[1:-1, j],
                    intrinsic
                )

    def get_option_price(self, S0):

        if self.barrier_price is None:
            i = int(S0 / self.delta_s)
        else:
            if self.option_basic_type == OptionBasicType.CALL:
                i = int(S0 / self.delta_s)
            else:
                i = int(
                    (S0 - self.barrier_price)
                    / self.delta_s
                )

        j = self.M

        if i < 0 or i > self.N:
            raise ValueError("S0 outside grid")

        return self.grid[i, j]

    def get_option_prices_T0(self):

        return self.S_values, self.grid[:, -1]